###Ingest COU_transactions csv file   
1.read file using dataframe reader api   
2.add metadata columns - source file,ingestion timestamp     
3.write bronze tables using dataframe write api

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment_config


In [0]:
%run ../00-common/02.bronze_functions

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/COU_transactions/"

In [0]:
table_name = f"{catalog_name}.{bronze_schema}.COU_transactions"

In [0]:
from pyspark.sql.types import *

In [0]:
COU_schema = StructType([
  StructField('TransactionID', StringType()),
  StructField('BillerID', StringType()),
  StructField('CustomerID', StringType()),
  StructField('Amount', DoubleType()),
  StructField('Status', StringType()),
  StructField('BillerReferenceNo', StringType()),
  StructField('BankReferenceNo', StringType()),
  StructField('TransactionDate', TimestampType())
])

In [0]:
COU_df = spark.read.format('csv') \
                .option('header', True) \
                .schema(COU_schema) \
                .option('mode', 'FAILFAST') \
                .load(source_file)

In [0]:
display(COU_df)

In [0]:
COU_audit = add_ingestion_metadata(COU_df)

In [0]:
COU_final= COU_audit.withColumn("batch_id", F.lit(v_batch_id))

In [0]:
COU_final.write.mode('overwrite').partitionBy('batch_id').option('replaceWhere',f"batch_id = '{v_batch_id}'").saveAsTable(table_name)